# Assignment 2: OB journal articles from Crossref

This notebook follows `tutorials/crossref_api.ipynb`: find journals and their
ISSNs, request their works, parse the JSON, and save a pandas DataFrame to CSV.
It collects the following five journals, from January 1, 2015 through the date
you run it:

- Journal of Applied Psychology (JAP)
- Personnel Psychology (Ppsych)
- Academy of Management Journal (AMJ)
- Academy of Management Review (AMR)
- Journal of Organizational Behavior (JOB)

Unlike the tutorial's single-page example, this notebook follows cursor pages
until each query is exhausted. Both print and online ISSNs are queried, then
duplicate DOIs are removed. Coverage is limited to records deposited in Crossref;
`journal-article` can include editorials or other content publishers classify
as articles. Missing abstracts and counts are left blank rather than invented.

**Run the cells in order when ready. None have been executed.** Statistical
analyses will be done separately; this notebook only creates `ten_years_api.csv`.

If imports fail, run this command in a separate notebook code cell:

```python
%pip install curl-cffi beautifulsoup4 pandas
```


## 1. Imports and settings

Replace `your_email@ucf.edu` with your email, as in the tutorial. Crossref uses
the contact information to identify API clients. No API key is needed.
The end date is the local date when this cell runs.


In [1]:
from datetime import date
from pathlib import Path
from time import sleep
import re

from bs4 import BeautifulSoup
from curl_cffi import requests
import pandas as pd

YOUR_EMAIL = "sh094418@ucf.edu"
START_DATE = "2015-01-01"
END_DATE = date.today().isoformat()
PAGE_SIZE = 500
API_BASE = "https://api.crossref.org"

JOURNALS = [
    "Journal of Applied Psychology",
    "Personnel Psychology",
    "Academy of Management Journal",
    "Academy of Management Review",
    "Journal of Organizational Behavior",
]
COLUMNS = [
    "doi", "title", "n_refs", "n_authors", "n_cites",
    "url", "journal", "abstract", "pub_year",
]

if "your_email" in YOUR_EMAIL or "@" not in YOUR_EMAIL:
    raise ValueError("Replace YOUR_EMAIL with your actual email address, then rerun this cell.")

print(f"Collecting {len(JOURNALS)} journals from {START_DATE} through {END_DATE}.")


## 2. Request JSON from the API

This helper adds the email to each request, checks HTTP errors, and retries
temporary connection errors, rate limits (429), and server errors (5xx).
Requests are spaced out to reduce load on Crossref. If retries fail, execution
stops rather than silently treating an incomplete download as complete.


In [2]:
def get_message(endpoint, params=None):
    query = dict(params or {})
    query["mailto"] = YOUR_EMAIL
    headers = {"User-Agent": f"MAN7916-Assignment2/1.0 (mailto:{YOUR_EMAIL})"}

    for attempt in range(5):
        sleep(1)
        try:
            response = requests.get(
                API_BASE + endpoint, params=query, headers=headers, timeout=90
            )
        except requests.exceptions.RequestException:
            if attempt == 4:
                raise
            delay = 2 ** (attempt + 1)
            print(f"Connection error; retrying in {delay} seconds.")
            sleep(delay)
            continue

        if response.status_code == 429 or 500 <= response.status_code < 600:
            if attempt == 4:
                response.raise_for_status()
            retry_after = response.headers.get("Retry-After", "")
            delay = int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1)
            print(f"HTTP {response.status_code}; retrying in {delay} seconds.")
            sleep(delay)
            continue

        response.raise_for_status()
        payload = response.json()
        if payload.get("status") != "ok":
            raise RuntimeError(f"Unexpected Crossref response for {endpoint}")
        return payload["message"]

    raise RuntimeError(f"Could not retrieve {endpoint}")


## 3. Find each journal's ISSNs

Like the tutorial, search `/journals` and match the full journal title rather
than accepting the first search result.


In [3]:
def normalize_title(title):
    return re.sub(r"[^a-z0-9]", "", title.casefold().replace("&", "and"))


def find_journal_issns(journal_title):
    message = get_message("/journals", {"query": journal_title, "rows": 100})
    matches = [
        item for item in message["items"]
        if normalize_title(item.get("title", "")) == normalize_title(journal_title)
    ]
    issns = sorted({issn for item in matches for issn in item.get("ISSN", [])})
    if not issns:
        candidates = [item.get("title", "") for item in message["items"][:10]]
        raise ValueError(f"No exact ISSN match for {journal_title}. Candidates: {candidates}")
    return issns


journal_issns = {}
for journal in JOURNALS:
    journal_issns[journal] = find_journal_issns(journal)
    print(f"{journal}: {', '.join(journal_issns[journal])}")


Journal of Applied Psychology: 0021-9010, 1939-1854, 2645-3541
Personnel Psychology: 0031-5826, 1744-6570
Academy of Management Journal: 0001-4273, 1948-0989
Academy of Management Review: 0363-7425, 1930-3807
Journal of Organizational Behavior: 0894-3796, 1099-1379


## 4. Parse the requested fields and retrieve all pages

Crossref supplies titles as lists, authors as lists of dictionaries, and dates
as nested `date-parts`. We use the first title and the year from `published`,
falling back to print or online publication dates if needed. Abstracts may
contain JATS/XML tags; BeautifulSoup removes the tags while keeping the text.
The `journal` column uses the matched journal name from our settings.

`n_refs` uses `reference-count` (or `references-count`, used in the tutorial).
`n_cites` is Crossref's `is-referenced-by-count`, not a citation count from other
databases. Missing numeric metadata remains blank; a recorded zero stays zero.
`n_authors` counts deposited authors when an author list is present.

Cursor pagination starts at `*` and follows `next-cursor`. A page with fewer
than `PAGE_SIZE` results ends that ISSN query. Progress messages show each page
and the number of unique DOIs collected so far.


In [4]:
def clean_text(value):
    return BeautifulSoup(value or "", "html.parser").get_text(" ", strip=True)


def publication_year(article):
    for field in ("published", "published-print", "published-online", "issued"):
        parts = (article.get(field) or {}).get("date-parts", [])
        if parts and parts[0] and parts[0][0] is not None:
            return int(parts[0][0])
    return None


def parse_article(article, journal):
    doi = article.get("DOI")
    if not doi:
        raise ValueError(f"A Crossref record in {journal} has no DOI.")
    titles = article.get("title") or []
    authors = article.get("author")
    return {
        "doi": doi.strip().lower(),
        "title": clean_text(titles[0]) if titles else "",
        "n_refs": article.get("reference-count", article.get("references-count")),
        "n_authors": len(authors) if authors is not None else None,
        "n_cites": article.get("is-referenced-by-count"),
        "url": article.get("URL") or f"https://doi.org/{doi}",
        "journal": journal,
        "abstract": clean_text(article.get("abstract")),
        "pub_year": publication_year(article),
    }


def iter_journal_articles(issn):
    cursor = "*"
    page_number = 0
    seen_pages = set()
    while True:
        message = get_message(f"/journals/{issn}/works", {
            "filter": (
                f"type:journal-article,from-pub-date:{START_DATE},"
                f"until-pub-date:{END_DATE}"
            ),
            "rows": PAGE_SIZE,
            "cursor": cursor,
        })
        items = message["items"]
        page_number += 1
        print(f"  ISSN {issn}, page {page_number}: {len(items)} records "
              f"(API reports {message['total-results']} matching records).")
        if not items:
            break

        signature = tuple(item.get("DOI", "").lower() for item in items)
        if signature in seen_pages:
            raise RuntimeError(f"Repeated result page for ISSN {issn}; download stopped.")
        seen_pages.add(signature)
        yield from items

        if len(items) < PAGE_SIZE:
            break
        next_cursor = message.get("next-cursor")
        if not next_cursor:
            raise RuntimeError(f"Missing pagination cursor for ISSN {issn}; download may be incomplete.")
        # Crossref may reuse the same cursor string as it advances its server-side state.
        cursor = next_cursor


## 5. Collect the five journals into one DataFrame

Each DOI appears once, even if returned under both print and online ISSNs.
If a second copy has a field that was missing in the first, use that value.
A journal with no results raises an error so it cannot be overlooked.


In [5]:
records_by_doi = {}

for journal, issns in journal_issns.items():
    print(f"\nCollecting {journal}")
    journal_dois = set()
    for issn in issns:
        for article in iter_journal_articles(issn):
            record = parse_article(article, journal)
            doi = record["doi"]
            journal_dois.add(doi)
            if doi not in records_by_doi:
                records_by_doi[doi] = record
            else:
                existing = records_by_doi[doi]
                if existing["journal"] != journal:
                    raise ValueError(f"DOI {doi} unexpectedly belongs to two selected journals.")
                for field in COLUMNS:
                    if existing[field] is None or existing[field] == "":
                        existing[field] = record[field]
        print(f"  Unique articles collected for this journal: {len(journal_dois)}")
    if not journal_dois:
        raise ValueError(f"No articles retrieved for {journal}; check the ISSNs and dates.")
    print(f"Finished {journal}: {len(journal_dois)} unique articles.")

article_df = pd.DataFrame(records_by_doi.values(), columns=COLUMNS)
for column in ["n_refs", "n_authors", "n_cites", "pub_year"]:
    article_df[column] = pd.to_numeric(article_df[column], errors="raise").astype("Int64")
article_df = article_df.sort_values(["journal", "pub_year", "doi"]).reset_index(drop=True)

print(f"\nTotal: {len(article_df)} unique articles.")
print(article_df.groupby("journal").size().to_string())
print(f"Articles with abstracts: {article_df['abstract'].ne('').sum()}")
print("Missing numeric metadata:")
print(article_df[["n_refs", "n_authors", "n_cites", "pub_year"]].isna().sum())
article_df.head()



  ISSN 0021-9010, page 1: 500 records (API reports 1637 matching records).
  ISSN 0021-9010, page 2: 500 records (API reports 1637 matching records).
  ISSN 0021-9010, page 3: 500 records (API reports 1637 matching records).
  ISSN 0021-9010, page 4: 137 records (API reports 1637 matching records).
  Unique articles collected for this journal: 1637
  ISSN 1939-1854, page 1: 500 records (API reports 1637 matching records).
  ISSN 1939-1854, page 2: 500 records (API reports 1637 matching records).
  ISSN 1939-1854, page 3: 500 records (API reports 1637 matching records).
  ISSN 1939-1854, page 4: 137 records (API reports 1637 matching records).
  Unique articles collected for this journal: 1637
  ISSN 2645-3541, page 1: 73 records (API reports 73 matching records).
  Unique articles collected for this journal: 1710
Finished Journal of Applied Psychology: 1710 unique articles.

  ISSN 0031-5826, page 1: 500 records (API reports 645 matching records).
  ISSN 0031-5826, page 2: 145 records

,doi,title,n_refs,n_authors,n_cites,url,journal,abstract,pub_year
0,10.5465/amj.2010.0837,The Person–Situation Debate Revisited: Effect ...,233,2,598,https://doi.org/10.5465/amj.2010.0837,Academy of Management Journal,,2015
1,10.5465/amj.2011.0214,Inherently Relational: Interactions between Pe...,117,5,39,https://doi.org/10.5465/amj.2011.0214,Academy of Management Journal,,2015
2,10.5465/amj.2011.0276,I Need Time! Exploring Pathways to Compliance ...,72,4,166,https://doi.org/10.5465/amj.2011.0276,Academy of Management Journal,,2015
3,10.5465/amj.2011.0347,"Ostracism, Self-Esteem, and Job Performance: W...",90,4,244,https://doi.org/10.5465/amj.2011.0347,Academy of Management Journal,,2015
4,10.5465/amj.2011.0727,"“I Put in Effort, Therefore I Am Passionate”: ...",78,5,259,https://doi.org/10.5465/amj.2011.0727,Academy of Management Journal,,2015


## 6. Save `ten_years_api.csv` in `assignment_2`


UTF-8 with a BOM helps Excel display special characters. The folder lookup
supports starting Jupyter from this assignment directory or elsewhere inside
the repository. Close an existing copy in Excel before overwriting it.



References: [Crossref REST API guidance](https://www.crossref.org/documentation/retrieve-metadata/rest-api/tips-for-using-the-crossref-rest-api/)
and [API documentation](https://api.crossref.org/swagger-ui/index.html).


In [6]:
def find_assignment_folder():
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        if (folder / "ten_years_api.ipynb").is_file():
            return folder
        candidate = folder / "assignments" / "submissions" / "assignment_2"
        if (candidate / "ten_years_api.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter from within the repository folder.")


if article_df.empty or set(article_df["journal"]) != set(JOURNALS):
    raise ValueError("Results must include all five journals before saving.")
if article_df["doi"].duplicated().any():
    raise ValueError("Duplicate DOIs found; review the collection step.")

output_file = find_assignment_folder() / "ten_years_api.csv"
article_df.to_csv(output_file, columns=COLUMNS, index=False, encoding="utf-8-sig")
print(f"Saved {len(article_df)} articles to {output_file}")


Saved 4980 articles to C:\Research\Subjects\content analysis\MAN7916-main-testing\assignments\submissions\assignment_2\ten_years_api.csv
